This will:

🧹 Clean and chunk the raw scraped website content

🧠 Embed the chunks using a strong embedding model (e.g., BGE-small-en)

🗂️ Store the vectors in a FAISS index for fast similarity search


🧠 Purpose:
This notebook handles data preparation for RAG by:

🧹 Cleaning + chunking your raw scraped university website text

🧠 Embedding each chunk into a vector

🗂️ Saving the vectors in a FAISS index for fast semantic retrieval during chatbot inference

It is used by:

rag_sft_chatbot.ipynb

rag_dapt_sft_chatbot.ipynb

Step 2: Imports
Brings in:

faiss for similarity search indexing

sentence_transformers for embeddings

pickle for saving metadata

tqdm for progress bars


In [1]:
# 📌 Step 2: Imports
import os
from pathlib import Path
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Step 3: Define Paths
input_path: your scraped raw text

index_save_path: folder for FAISS and chunks

Creates the output directory if it doesn't exist

In [2]:
# 📌 Step 3: Define Paths
input_path = Path("../data/full_university_data.txt")
index_save_path = Path("../checkpoints/faiss_embeddings/")
index_save_path.mkdir(parents=True, exist_ok=True)

Step 4: Load and Chunk Text
Reads full text from the university website

Splits on double newline (\n\n) → common heuristic for paragraph breaks

Filters out small junk chunks (<50 characters)

✅ Result: clean, paragraph-style chunks stored as chunks[]

In [3]:
# 📌 Step 4: Load and Chunk Text
with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Simple paragraph-based chunking
chunks = [chunk.strip() for chunk in raw_text.split("\n\n") if len(chunk.strip()) > 50]
print(f"📄 Total Chunks: {len(chunks)}")

📄 Total Chunks: 439


Step 5: Load Embedding Model
Loads BAAI/bge-small-en-v1.5, which was later also used at inference time

Embeds English sentences into a dense 384-dim space

In [ ]:
# 📌 Step 5: Load Embedding Model
model = SentenceTransformer("BAAI/bge-small-en-v1.5") 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\berfi\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 6: Embed Chunks
Converts all text chunks into embeddings

Uses model.encode(..., show_progress_bar=True)

In [6]:
# 📌 Step 6: Embed Chunks
embeddings = model.encode(chunks, show_progress_bar=True)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step 7: Build FAISS Index
Gets dimensionality from first vector

Uses IndexFlatL2 (simple + fast) for similarity search

Adds all vectors

In [7]:
# 📌 Step 7: Build FAISS Index
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

Step 8: Save Index + Metadata
Writes the FAISS index to: university_index.faiss

Dumps the chunks (texts) to: university_chunks.pkl

These files are later loaded during inference to perform context-based search before response generation.

In [8]:
# 📌 Step 8: Save FAISS Index and Metadata
faiss.write_index(index, str(index_save_path / "university_index.faiss"))

with open(index_save_path / "university_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("✅ FAISS index and chunks saved to:", index_save_path)

✅ FAISS index and chunks saved to: ..\checkpoints\faiss_embeddings


✅ Pipeline Position

Component	Used by Pipelines
FAISS index	✅ P4 (Full RAG) and rag_sft_chatbot.ipynb
Chunks.pkl	Same — used to fetch original matched context